In [ ]:

# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------

import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ---------------------------------------------------------------------
# INICIANDO SPARK
# ---------------------------------------------------------------------

spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics_Gold05")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# IDENTIFICAR ROOT DO PROJETO
# ---------------------------------------------------------------------

PROJECT_ROOT = Path(__file__).resolve().parents[3]

print("PROJECT_ROOT:")
print(PROJECT_ROOT)


# ---------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 05 - Qual é o índice de adoção de Inteligência Artificial e seu impacto?
# ---------------------------------------------------------------------

caminho_gold_05 = PROJECT_ROOT / "Gold" / "perguntas_negocio" / "gold_05_adocao_ia"

print("\nCaminho Gold 05:")
print(caminho_gold_05)


# ---------------------------------------------------------------------
# BUSCA CSVs GERADOS PELO SPARK
# ---------------------------------------------------------------------

arquivos_gold_05 = [
    str(arquivo) for arquivo in caminho_gold_05.glob("part-*.csv")
]

print("\nArquivos encontrados:")
print(arquivos_gold_05)


# ---------------------------------------------------------------------
# CARREGAR GOLD 05
# ---------------------------------------------------------------------

"""
A Gold 05 reúne dois formatos de informação na mesma estrutura: distribuições de respostas, representadas por variavel, valor, contagem e total_respondentes, e indicadores de adoção, representados por categoria, opcao, elegiveis, selecionaram e pct_adocao. Essa diferença de estrutura explica parte dos valores nulos observados posteriormente.
"""
df_ia = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_05)
    .withColumnRenamed("variavel_original", "variavel")
)


# ---------------------------------------------------------------------
# INSPEÇÃO INICIAL
# ---------------------------------------------------------------------

"""
A inspeção inicial é realizada antes de qualquer recorte para validar a estrutura efetivamente entregue pela Gold 05. O notebook executado confirmou 76 registros e 11 colunas, permitindo identificar como os indicadores de IA foram organizados antes das análises posteriores.
"""
print("\n================ GOLD 05 ================\n")

df_ia.show(50, truncate=False)

print("\nSCHEMA:")
df_ia.printSchema()

print("\nQuantidade de linhas:")
print(df_ia.count())

print("\nColunas:")
print(df_ia.columns)


# ---------------------------------------------------------------------
# EDIÇÕES DISPONÍVEIS
# ---------------------------------------------------------------------

"""
A base contém as três edições utilizadas no projeto: 2023-2024, 2024-2025 e 2025-2026. A presença das três pesquisas permite avaliar evolução histórica, desde que cada indicador seja comparado apenas nas edições em que efetivamente existe.
"""
print("\n================ EDIÇÕES ================\n")

(
    df_ia
    .select("edicao")
    .distinct()
    .orderBy("edicao")
    .show(truncate=False)
)


# ---------------------------------------------------------------------
# VARIÁVEIS DISPONÍVEIS
# ---------------------------------------------------------------------

"""
A inspeção mostra que a única variável armazenada no formato de distribuição é ai_generativa_e_llm_e_uma_prioridade, disponível apenas em 2024-2025 e 2025-2026. Os demais registros possuem variavel nula porque representam indicadores estruturados por categoria e opção de adoção.
"""
print("\n================ VARIÁVEIS ================\n")

(
    df_ia
    .select("variavel")
    .distinct()
    .orderBy("variavel")
    .show(200, truncate=False)
)


# ---------------------------------------------------------------------
# QUANTIDADE DE REGISTROS POR EDIÇÃO E VARIÁVEL
# ---------------------------------------------------------------------

"""
Cada edição possui 22 registros no formato de categoria e opção. Além disso, 2024-2025 e 2025-2026 possuem cinco registros adicionais referentes às cinco respostas possíveis sobre a prioridade da IA generativa na empresa. Essa diferença deve ser considerada ao comparar o volume de linhas entre as edições.
"""
print("\n================ REGISTROS POR VARIÁVEL ================\n")

(
    df_ia
    .groupBy("edicao", "variavel")
    .agg(F.count("*").alias("qtd_linhas"))
    .orderBy("edicao", "variavel")
    .show(500, truncate=False)
)


# ---------------------------------------------------------------------
# VALORES DISPONÍVEIS EM CADA VARIÁVEL
# ---------------------------------------------------------------------

"""
A pergunta sobre prioridade corporativa mantém as mesmas cinco possibilidades de resposta em 2024-2025 e 2025-2026, permitindo comparação direta entre as duas edições. Nos outputs, a parcela que considera IA entre as principais prioridades passa de 31,1% para 36,8%, enquanto a resposta de que IA não é uma prioridade passa de 14,8% para 11,3%.
"""
print("\n================ VALORES POR VARIÁVEL ================\n")

(
    df_ia
    .select("edicao", "variavel", "valor")
    .distinct()
    .orderBy("variavel", "edicao", "valor")
    .show(1000, truncate=False)
)


# ---------------------------------------------------------------------
# CATEGORIAS DISPONÍVEIS
# ---------------------------------------------------------------------

"""
As três edições possuem indicadores de uso pessoal de IA, tipos de uso de IA nas empresas e motivos para não utilizar IA. A categoria de prioridade corporativa aparece somente a partir de 2024-2025, portanto não deve ser comparada com 2023-2024.
"""
print("\n================ CATEGORIAS ================\n")

(
    df_ia
    .select("edicao", "categoria")
    .distinct()
    .orderBy("edicao", "categoria")
    .show(500, truncate=False)
)


# ---------------------------------------------------------------------
# OPÇÕES DISPONÍVEIS
# ---------------------------------------------------------------------

"""
A inspeção das opções confirma que os indicadores de adoção permitem múltiplas seleções. Por isso, os percentuais das opções de uma mesma categoria podem ultrapassar 100% quando somados e devem ser interpretados individualmente sobre o número de elegíveis, e não como partes exclusivas de uma distribuição.
"""
print("\n================ OPÇÕES ================\n")

(
    df_ia
    .select("edicao", "categoria", "opcao")
    .filter(F.col("opcao").isNotNull())
    .distinct()
    .orderBy("edicao", "categoria", "opcao")
    .show(1000, truncate=False)
)


# ---------------------------------------------------------------------
# VERIFICAR NULOS
# ---------------------------------------------------------------------

"""
Os nulos encontrados são principalmente estruturais. Os 66 registros de adoção por categoria não utilizam variavel, valor, contagem, total_respondentes e pct_na_dimensao, enquanto os 10 registros da pergunta sobre prioridade corporativa não utilizam opcao, elegiveis, selecionaram e pct_adocao. Portanto, esses nulos não são tratados automaticamente como ausência ou erro de preenchimento.
"""
print("\n================ NULOS ================\n")

df_ia.select(
    [
        F.sum(
            F.when(F.col(coluna).isNull(), 1).otherwise(0)
        ).alias(coluna)
        for coluna in df_ia.columns
    ]
).show(truncate=False)


# ---------------------------------------------------------------------
# VERIFICAR TOTAIS DE RESPONDENTES
# ---------------------------------------------------------------------

"""
O campo total_respondentes está disponível somente para a distribuição de prioridade corporativa, com 1.045 respondentes em 2024-2025 e 652 em 2025-2026. Para as categorias de adoção, o denominador correto está no campo elegiveis, evitando utilizar um total geral inadequado para perguntas respondidas por públicos diferentes.
"""
print("\n================ TOTAL DE RESPONDENTES ================\n")

(
    df_ia
    .select("edicao", "variavel", "total_respondentes")
    .distinct()
    .orderBy("edicao", "variavel")
    .show(500, truncate=False)
)


# ---------------------------------------------------------------------
# VERIFICAR DADOS DE ADOÇÃO DE IA
# ---------------------------------------------------------------------

"""
Os outputs mostram mudança relevante no tipo de acesso às ferramentas de IA. No uso pessoal, soluções gratuitas passam de 63,7% em 2023-2024 para 30,5% em 2025-2026, enquanto soluções pagas pela empresa avançam de 6,4% para 42,4% e o uso de copilots passa de 11,8% para 29,8%. No mesmo período, a opção de não utilizar soluções de IA cai de 19,7% para 2,1%.
"""
print("\n================ ADOÇÃO DE IA ================\n")

(
    df_ia
    .select(
        "edicao",
        "categoria",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .filter(F.col("opcao").isNotNull())
    .orderBy("edicao", "categoria", F.desc("pct_adocao"))
    .show(1000, truncate=False)
)


# ---------------------------------------------------------------------
# VERIFICAR DISTRIBUIÇÃO COMPLETA
# ---------------------------------------------------------------------

"""
A visualização conjunta é mantida para validar como os dois formatos de indicador coexistem na Gold. Entre os usos empresariais, o uso independente e descentralizado permanece como a opção mais frequente, passando de 43,7% para 48,8%, enquanto o direcionamento centralizado cresce de 10,9% para 36,3%.

Os motivos para não utilizar IA também mudam ao longo do período. Em 2025-2026, falta de expertise ou recursos aparece como principal barreira, com 38,8%, seguida por dados da empresa ainda não preparados para IA generativa, com 36,8%, e falta de compreensão dos casos de uso, com 32,0%.
"""
print("\n================ DISTRIBUIÇÃO COMPLETA ================\n")

(
    df_ia
    .select(
        "edicao",
        "variavel",
        "valor",
        "contagem",
        "total_respondentes",
        "pct_na_dimensao",
        "categoria",
        "opcao",
        "elegiveis",
        "selecionaram",
        "pct_adocao"
    )
    .orderBy(
        "edicao",
        "variavel",
        F.desc("contagem")
    )
    .show(1000, truncate=False)
)